# Baseline model: the bar every other model must clear

**What a baseline is.** The simplest sensible prediction, built with no risk factors at all. If a Poisson GLM or a boosting model cannot beat it, it has learned nothing. It also gives us a reference *score*, so that later results mean something.

**Our baseline:** every policy gets the same claim rate (claims per exposure-year), scaled by its own exposure. A policy observed for half a year is expected to have half the claims of a full-year policy.

**Rules of this notebook**
* Only the **training set** is used. The test set stays locked until the very end of the project.
* Models are compared with **grouped 5-fold cross-validation** inside the training set (same folds for every model, saved in `train_folds.csv`).
* The metric is the **Poisson deviance** (lower is better), and calibration is always checked by **exposure band**.
* All shared logic (folds, deviance, cross-validation loop, calibration table) lives in `src/evaluation.py`, so every later model is scored by identical code.

In [3]:
# Cell 1: setup and training data
# (Magics must stay alone on their line: no comments after them.)
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")      # lets the notebook import our project code from src/
import numpy as np
import pandas as pd

from src.evaluation import (
    load_train,                  # training policies + the frozen Fold column
    cross_validate,              # the shared CV loop: fits on 4 folds, predicts the 5th, scores it
    poisson_deviance,            # the metric
    calibration_by_exposure,     # actual vs predicted claims per exposure band
    exposure_band_reference,     # a yardstick (NOT a model), used in the last step
    N_FOLDS, EXPOSURE_EDGES,
)

train = load_train()
print(train.shape)               # your run: (541432, 23)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(541432, 23)


## Step 1: the folds
Cross-validation splits the training set into 5 parts (folds). Each model is trained on 4 and scored on the 5th, five times, so every policy is predicted by a model that never saw it.

**Why *grouped* folds?** About 26% of rows are near-duplicate pairs (same features, consecutive IDs, exposure adding up to about one year: very likely one policy-year split into two records). If two twins land in different folds, a flexible model can memorise one and "predict" the other, which flatters the score. So every group of identical-feature policies is kept inside one fold. This was decided and saved once; here we only verify it.

In [4]:
# Cell 2: verify the folds (read-only: nothing is created or changed here)
# 1) Leak check: a feature-profile group must never appear in two folds.
assert train.groupby("Group")["Fold"].nunique().max() == 1
assert sorted(train["Fold"].unique()) == list(range(N_FOLDS))

# 2) Balance: folds should hold ~20% of the policies and have a similar frequency (claims / exposure).
fb = train.groupby("Fold").agg(policies=("IDpol", "size"), exposure=("Exposure", "sum"), claims=("ClaimNb", "sum"))
fb["share"] = fb["policies"] / len(train)
fb["freq"] = fb["claims"] / fb["exposure"]
print(fb.round(4))

# Your run: shares 0.1992 to 0.2004; frequencies 0.1009, 0.1008, 0.1022, 0.1011, 0.0994.
# With ~5,800 claims per fold the natural noise on the frequency is about +-1.3% (+-0.0013), so these gaps are chance.

      policies    exposure  claims   share    freq
Fold                                              
0       108528  57157.9490    5769  0.2004  0.1009
1       108190  57096.3277    5758  0.1998  0.1008
2       108348  57106.7558    5834  0.2001  0.1022
3       108519  57298.0927    5794  0.2004  0.1011
4       107847  57135.5788    5677  0.1992  0.0994


## Step 2: the metric, Poisson deviance
Claim counts are small non-negative integers, with 95% zeros. Accuracy is useless here (predicting "no claim" for everyone is 95% accurate), and RMSE measures absolute errors, which is a poor guide for counts that are mostly zeros. The Poisson deviance compares the predicted expected count `mu` with the actual count `y`:

`deviance = 2 * mean( y*log(y/mu) - (y - mu) )`   (the `y*log(y/mu)` term is 0 when `y = 0`)

It is 0 for a perfect prediction and punishes errors **relative to the expected size**: predicting 0.01 claims when one happened costs far more than predicting 2.

In [5]:
# Cell 3: build intuition for the metric with cases you can check by hand
# These first three also run as asserts every time src/evaluation.py is imported.
print(poisson_deviance([0], [0.5]))   # 1.0    = 2*(0 - (0 - 0.5)): no claim, but 0.5 predicted
print(poisson_deviance([2], [2]))     # 0.0    perfect prediction
print(poisson_deviance([1], [2]))     # 0.6137 = 2*(ln(0.5) + 1): the term y*log(y/mu) is exercised

# The cost of missing ONE real claim depends on how small the prediction was (squared error for contrast):
print("\npredicted mu | squared error | deviance   (one claim happened)")
for mu in [0.01, 0.1, 0.5, 1.0, 2.0]:
    print(f"{mu:>12} | {(1 - mu) ** 2:>13.3f} | {poisson_deviance([1], [mu]):.3f}")
# expected deviances: 7.230, 2.805, 0.386, 0.000, 0.614

# Why the average deviance is dominated by claimants: a policy with NO claim and mu = 0.05 costs only 2*mu,
# while a policy WITH a claim and the same mu costs about 40 times more.
print("\nno claim, mu=0.05:", round(poisson_deviance([0], [0.05]), 3))       # 0.1
print("one claim, mu=0.05:", round(poisson_deviance([1], [0.05]), 3))         # 4.091

1.0
0.0
0.6137056388801094

predicted mu | squared error | deviance   (one claim happened)
        0.01 |         0.980 | 7.230
         0.1 |         0.810 | 2.805
         0.5 |         0.250 | 0.386
         1.0 |         0.000 | 0.000
         2.0 |         1.000 | 0.614

no claim, mu=0.05: 0.1
one claim, mu=0.05: 4.091


## Step 3: the baseline model
For each fold: learn **one rate** from the 4 training folds, `rate = total claims / total exposure` (sum over sum, so a one-day policy cannot distort it), then predict for every policy in the 5th fold `expected claims = rate * exposure`.

`cross_validate` takes a function `fit_predict(train_part, valid_part)` that returns expected claim counts. The same loop will score the GLM and boosting models.

In [6]:
# Cell 4: the baseline, scored with grouped cross-validation
def baseline(tr, va):
    """One constant claim rate learned on the training folds only, scaled by each policy's exposure."""
    rate = tr["ClaimNb"].sum() / tr["Exposure"].sum()      # exposure-weighted rate
    return (va["Exposure"] * rate).to_numpy()              # expected claims = rate x exposure

oof_b, s_b = cross_validate(train, baseline)               # oof_b: out-of-fold predictions; s_b: deviance per fold

for k in range(N_FOLDS):
    tr_k = train[train["Fold"] != k]
    print(f"fold {k}: rate = {tr_k['ClaimNb'].sum() / tr_k['Exposure'].sum():.4f} | deviance = {s_b[k]:.4f}")
print("\nbaseline mean deviance:", round(s_b.mean(), 4), "+/-", round(s_b.std(), 4))

# Your run: rates 0.1009, 0.1009, 0.1006, 0.1008, 0.1013
#           deviances 0.3291, 0.3305, 0.3330, 0.3349, 0.3285 -> mean 0.3312 +/- 0.0024
assert np.isclose(s_b.mean(), 0.3312, atol=1e-4), "does not reproduce the reference baseline"
# The fold-to-fold spread (0.0024) is the NOISE of a single score. We never compare models on their means alone:
# every model is scored on the SAME folds, so we compare them fold by fold (Step 5).

fold 0: rate = 0.1009 | deviance = 0.3291
fold 1: rate = 0.1009 | deviance = 0.3305
fold 2: rate = 0.1006 | deviance = 0.3330
fold 3: rate = 0.1008 | deviance = 0.3349
fold 4: rate = 0.1013 | deviance = 0.3285

baseline mean deviance: 0.3312 +/- 0.0024


## Step 4: calibration by exposure band
A good score is not enough: a model can be right on average and badly wrong for a whole group. For each exposure band we compare the **actual** number of claims with the **predicted** one. `actual / predicted` above 1 means the group is under-predicted.

In [7]:
# Cell 5: calibration of the baseline
print("overall actual/predicted:", round(train["ClaimNb"].sum() / oof_b.sum(), 4), "\n")      # ~1.0 by construction
print(calibration_by_exposure(train, oof_b))



overall actual/predicted: 1.0 

             policies  actual  predicted  actual/predicted
Exposure                                                  
(0.0, 0.1]     102054    2785      607.9             4.581
(0.1, 0.25]     76054    2890     1404.8             2.057
(0.25, 0.5]    105108    5212     4149.6             1.256
(0.5, 0.75]     73967    4822     4725.0             1.021
(0.75, 1.0]    184249   13123    17944.6             0.731


## Step 5: a yardstick, not a model
How much of the score is just *exposure structure*? We predict with the exposure band alone (one rate per band). **This is not a candidate model**: when pricing a new policy its future duration is unknown, so exposure can never be an input. It only tells us how to read later improvements: gains smaller than this gap may just reflect features that correlate with short exposure (new vehicles, for example).

In [ ]:
# Cell 6: exposure-band yardstick and paired comparison
oof_e, s_e = cross_validate(train, exposure_band_reference)

gap = s_b - s_e                      # same folds, so we can compare fold by fold ("paired")
print("baseline        :", s_b.round(4), "| mean", round(s_b.mean(), 4))
print("exposure yardstick:", s_e.round(4), "| mean", round(s_e.mean(), 4))
print("gap per fold    :", gap.round(4), "| mean", round(gap.mean(), 4), "| spread", round(gap.std(), 4))
print("relative gain   :", f"{gap.mean() / s_b.mean():.1%}")



baseline        : [0.3291 0.3305 0.333  0.3349 0.3285] | mean 0.3312
exposure yardstick: [0.3165 0.318  0.3199 0.3215 0.3153] | mean 0.3183
gap per fold    : [0.0126 0.0124 0.0131 0.0133 0.0132] | mean 0.0129 | spread 0.0003
relative gain   : 3.9%


## Summary

| Quantity | Value |
|---|---|
| Baseline mean deviance (grouped 5-fold CV, training set) | **0.3312** +/- 0.0024 |
| Exposure-band yardstick (not a model) | 0.3183 (gap 0.0129, in every fold) |
| Overall calibration | 1.0 on average, but 4.6x under-predicted for exposure <= 0.1 and 0.73x for full-year policies |

**Next:** the Poisson GLM, scored by the same `cross_validate`. It must beat 0.3312 in every fold, and we will check how its calibration by exposure band compares with the table above.